# 04 · Reproduce AdaNeg (NeurIPS 2024), with NegLabel and MCM baselines

**Acceptance (pre-G1 plan):** Four-OOD FPR95 within **±1** of **18.92** (paper; the repository's own log
shows 17.95) and OpenOOD near-OOD close to **67.51**.

AdaNeg runs as in `scripts/ood/adaneg/imagenet.sh` (memory 10 per label, threshold 0.5, gap 0.5,
λ = 0.1, 5 groups, random permutation, `combine` score). Two precisions:

* **fp16** — bit-exact emulation of the official GPU run. Its per-sample entropy is NaN in fp16, so
  memory slots are never *replaced*: each label keeps its first 9 confident features (slot 0 holds its
  text feature). This is what produced the published numbers.
* **fp32** — the entropy rule as described in the paper (replace the most uncertain slot).

**Settings:** GPU recommended (≈ 30 min). **Inputs:** `oodlab-code` + notebook 01's output
(+ notebook 02's output, optional, for the final leaderboard).

In [ ]:
# ── Set-up 1/2: find the oodlab code (attached as a private Kaggle dataset) ─────────
import os, sys

def find_oodlab(roots, max_depth=4):
    """Bounded breadth-first search for a folder containing oodlab/__init__.py.
    Never walks image folders (the old notebook hung on a recursive glob over ImageNet)."""
    skip = {"ILSVRC", "train", "val", "test", "images", "__pycache__", "cache", "results", "logs"}
    for root in [r for r in roots if r and os.path.isdir(r)]:
        frontier = [(root, 0)]
        while frontier:
            d, depth = frontier.pop(0)
            if os.path.isfile(os.path.join(d, "oodlab", "__init__.py")):
                return d
            if depth < max_depth:
                try:
                    subs = sorted(e.path for e in os.scandir(d)
                                  if e.is_dir(follow_symlinks=False) and e.name not in skip)
                except OSError:
                    continue
                frontier += [(s, depth + 1) for s in subs]
    return None

ROOTS = [os.environ.get("OODLAB_CODE"), "/kaggle/input", "/kaggle/working", os.getcwd()]
CODE_ROOT = find_oodlab(ROOTS)
if CODE_ROOT is None:   # the dataset was uploaded as a zip that Kaggle did not unpack -> unpack it ourselves
    import zipfile
    for root in [r for r in ROOTS[:2] if r and os.path.isdir(r)]:
        for dp, dn, fns in os.walk(root):
            dn[:] = [d for d in dn if d not in {"ILSVRC", "train", "val", "test", "images"}] if dp.count(os.sep) - root.count(os.sep) < 3 else []
            for fn in fns:
                if fn.startswith("oodlab") and fn.endswith(".zip"):
                    zipfile.ZipFile(os.path.join(dp, fn)).extractall("/kaggle/working/_oodlab_code")
    CODE_ROOT = find_oodlab(["/kaggle/working/_oodlab_code"])
if CODE_ROOT is None:
    raise FileNotFoundError("oodlab code not found: attach your private dataset with the code "
                            "(Add Input -> Your Datasets -> oodlab-code). See GUIDE.md, section 2.")
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)
import oodlab
print("oodlab", oodlab.__version__, "loaded from", CODE_ROOT)

In [ ]:
# ── Set-up 2/2: packages, folders, logging ─────────────────────────────────────────
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True, gdown=False)      # CLIP ships inside oodlab; this only adds small pip packages if missing
ctx = start("04_reproduce_adaneg", need_gpu=False)
log = ctx.log                                   # log.info(...) -> screen + /kaggle/working/logs/04_reproduce_adaneg.log
import numpy as np, pandas as pd, torch, time, json
from oodlab.report import show, save_tables, write_report

In [ ]:
from oodlab.methods import AdaNeg, AdaNegConfig, NegLabel, NegLabelConfig, MCM, MCMConfig
from oodlab.runner import Runner, summarize, compare_with_targets, acceptance_table
from oodlab.report import find_cache

BACKBONE = "ViT-B/16"
SEEDS = [0, 1, 2]
ADANEG_PRECISIONS = ["fp16", "fp32"] if ctx.device == "cuda" else ["fp32"]
PROTOCOLS = ["four_ood", "near_50k", "openood_v15"]
if ctx.smoke:
    SEEDS = [0, 1]

cache = find_cache(ctx, BACKBONE)
bank = cache.load_textbank()
LS = bank.logit_scale
runner = Runner(cache, device=ctx.device, logger=log)
runs = []

### 1 · Our AdaNeg / NegLabel == official on real features

In [ ]:
from oodlab.official.equivalence import compare_adaneg, compare_neglabel
from oodlab.stream import pair_stream
eq_sets = [n for n in ["inaturalist", "ninco", "sun", "places", "textures_all", "ssb_hard"] if cache.has(n)]
assert cache.has("imagenet_val_all") and eq_sets, f"cache incomplete: {cache.available()}"
ids, ood = cache.load_set("imagenet_val_all"), cache.load_set(eq_sets[0])
s = pair_stream(ids.subset(range(min(4000, len(ids)))), ood.subset(range(min(2000, len(ood)))), seed=0)
batches = [s.feats(a, b) for _, a, b, _ in s.batches()]
eq = {"adaneg fp32": compare_adaneg(bank.id_text.float(), bank.neg_text.float(), [b.float() for b in batches],
                                    AdaNegConfig(logit_scale=LS), device=ctx.device),
      "neglabel fp32": compare_neglabel(bank.id_text.float(), bank.neg_text.float(), [b.float() for b in batches],
                                        NegLabelConfig(logit_scale=LS), device=ctx.device)}
if ctx.device == "cuda":
    eq["adaneg fp16"] = compare_adaneg(bank.id_text.half(), bank.neg_text.half(), [b.half() for b in batches],
                                       AdaNegConfig(logit_scale=LS, emulate_fp16=True), device=ctx.device)
    eq["neglabel fp16"] = compare_neglabel(bank.id_text.half(), bank.neg_text.half(), [b.half() for b in batches],
                                           NegLabelConfig(logit_scale=LS, emulate_fp16=True), device=ctx.device)
show(pd.DataFrame(eq).T, f"official vs ours (real features: ImageNet + {eq_sets[0]})", digits=8)
bad = [k for k, v in eq.items() if v["max_abs_diff"] > (1e-4 if "fp32" in k else 2e-3) or v["pred_agreement"] < 0.999]
print("EQUIVALENCE OK" if not bad else f"EQUIVALENCE CHECK FAILED for {bad} - investigate before trusting results")

### 2 · AdaNeg

In [ ]:
for protocol in PROTOCOLS:
    if runner.missing(protocol):
        log.warning(f"skip {protocol}: missing {runner.missing(protocol)}"); continue
    for prec in ADANEG_PRECISIONS:
        m = AdaNeg(bank.id_text, bank.neg_text, AdaNegConfig(logit_scale=LS, emulate_fp16=(prec == "fp16")), device=ctx.device)
        runs.append(runner.evaluate(m, protocol, SEEDS, config=prec))

### 3 · Baselines: NegLabel (static negatives) and MCM (ID labels only)

MCM twice: as in its paper (softmax of cosine / T, T = 1) and as `scripts/ood/mcm/official.sh` runs it
(softmax of 100·cosine / τ with τ chosen by OpenOOD's automatic parameter search on the validation
split: 5,000 ImageNet + 1,763 OpenImage-O images, τ ∈ {0.5, 1, 1.5, 2, 3, 5}, best AUROC).

In [ ]:
MCM_TAU = None
if not runner.missing("openood_val"):
    MCM_TAU, aps = runner.tune_on_val(lambda t: MCM(bank.id_text_simple, MCMConfig.openood_vlm(t, LS), device=ctx.device),
                                      [0.5, 1, 1.5, 2.0, 3.0, 5.0])
    show(aps, f"MCM tau search on the OpenOOD validation split -> tau = {MCM_TAU}")

In [ ]:
for protocol in PROTOCOLS:
    if runner.missing(protocol):
        continue
    nl_prec = "fp16" if ctx.device == "cuda" else "fp32"
    runs.append(runner.evaluate(NegLabel(bank.id_text, bank.neg_text,
                                         NegLabelConfig(logit_scale=LS, emulate_fp16=(nl_prec == "fp16")), device=ctx.device),
                                protocol, SEEDS, config=f"g100/{nl_prec}"))
    runs.append(runner.evaluate(MCM(bank.id_text_simple, MCMConfig.paper(), device=ctx.device), protocol, SEEDS, config="paper_T1"))
    if MCM_TAU is not None:
        runs.append(runner.evaluate(MCM(bank.id_text_simple, MCMConfig.openood_vlm(MCM_TAU, LS), device=ctx.device),
                                    protocol, SEEDS, config=f"openood_aps_tau{MCM_TAU}"))
if not runs:
    raise RuntimeError(f"no protocol could run - cached sets: {cache.available()}; finish notebook 01 first")
df = pd.concat(runs, ignore_index=True)
summary = summarize(df)

In [ ]:
for method in ["adaneg", "neglabel", "mcm"]:
    for protocol in PROTOCOLS:
        c = compare_with_targets(summary, method, protocol)
        if len(c):
            show(c, f"{method} vs published: {protocol}")
acc_table = acceptance_table(summary)
show(acc_table[acc_table.method == "adaneg"], "pre-G1 acceptance (AdaNeg)")

### 4 · Leaderboard (adds TANL if notebook 02's output is attached)

In [ ]:
tanl_csv = oodlab.paths.find_file("tanl_summary.csv", [ctx.dirs.results, ctx.dirs.inputs], max_depth=5)
board = summary.copy()
if tanl_csv:
    board = pd.concat([board, pd.read_csv(tanl_csv)], ignore_index=True)
    log.info(f"added TANL results from {tanl_csv}")
board = board[board.key.str.startswith("mean:")]
pivot = board.pivot_table(index=["method", "config"], columns=["protocol", "key"], values="fpr95").round(2)
show(pivot.reset_index(), "FPR95 (OOD-positive, OpenOOD convention) - lower is better")
save_tables(ctx, adaneg_runs=df, adaneg_summary=summary, adaneg_acceptance=acc_table, leaderboard=pivot.reset_index())
write_report(ctx, "AdaNeg / NegLabel / MCM reproduction",
             [("Acceptance", acc_table)] +
             [(f"{m} vs published: {p}", compare_with_targets(summary, m, p)) for m in ["adaneg", "neglabel", "mcm"]
              for p in PROTOCOLS if len(compare_with_targets(summary, m, p))])
finish(ctx)